In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from deepagents import create_deep_agent, CompiledSubAgent
from langchain.agents import create_agent
from app.tools.data_tools import *
from app.tools.metrics_agent_tools import *
from app.agents.prompts.data_analist_prompt import DATA_ANALYST_PROMPT
from app.agents.prompts.data_engineer_prompt import DATA_ENGINEER_PROMPT
from app.agents.prompts.ml_analist_prompt import ML_ANALYST_PROMPT
from app.agents.prompts.orchestrator_prompt import ORCHESTRATOR_SYSTEM_PROMPT

In [ ]:
data_eng_tools = [query_sql_campaigns, traffic_source_by_campaign, query_mongo_requests]
ml_specialist_tools = [run_ml_inference_pipeline, get_dataset_health_check]
data_analyst_tools = [get_context_data, query_anomalous_ids]

## Instanciando cliente Langfuse

In [ ]:
from langfuse import Langfuse, get_client
import os

Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com" 
)

langfuse = get_client()
langfuse_handler = CallbackHandler()

In [ ]:
from langchain_openai import ChatOpenAI

llm_fast = ChatOpenAI(
      model="gpt-4.1-mini"
)

llm_reasoning = ChatOpenAI(
      model="gpt-4o"
)

In [ ]:
data_engineer_graph = create_agent(
    model=llm_fast,
    tools=[
        query_sql_campaigns, 
        traffic_source_by_campaign,
        query_mongo_requests
    ],
    prompt=DATA_ENGINEER_PROMPT
)

data_eng_subagent = CompiledSubAgent(
    name="data-engineer",
    description="Fetches raw HTTP logs from databases based on hashes or traffic sources. Call this FIRST.",
    runnable=data_engineer_graph
)

In [ ]:
ml_analyst_graph = create_agent(
    model=llm_fast,
    tools=[
        run_ml_inference_pipeline, 
        get_dataset_health_check
    ],
    prompt=ML_ANALYST_PROMPT
)

ml_analyst_subagent = CompiledSubAgent(
    name="ml-inference-specialist",
    # description=,
    runnable=ml_analyst_graph
)

In [ ]:
data_analyst_graph = create_agent(
    model=llm_reasoning,
    tools=[
        compare_mismatch_frequencies
    ],
    prompt=DATA_ANALYST_PROMPT
)

data_analyst_subagent = CompiledSubAgent(
    name="bot-data-analyst",
    # description=,
    runnable=data_analyst_graph
)

In [ ]:
subagents = [
    data_eng_subagent, 
    ml_analyst_subagent, 
    data_analyst_subagent
]

In [ ]:
agent = create_deep_agent(
    model=llm_reasoning, 
    tools=[],
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
    subagents=subagents
)

In [ ]:
response = agent.ainvoke()